---

### 🎓 **Professor**: Apostolos Filippas

### 📘 **Class**: AI Engineering

### 📋 **Topic**: You Can Just Build Things

🚫 **Note**: You are not allowed to share the contents of this notebook with anyone outside this class without written permission by the professor.

---

## Welcome!

In our firstfour lectures, we've covered how
1. We can call LLMs via APIs and get structured responses
2. We can build lexical search with BM25
3. We can build semantic search with embeddings
4. We can combine lexical and semantic search into hybrid search

Today you will put it all together by building a Retrieval Augmented Generation (RAG) system.
- This is a question-answering bot that can answer questions about Fordham University
- You will use real data scraped from the Fordham website.


Your RAG pipeline will look like this:

```
User Question
     ↓
1. RETRIEVE: Find relevant documents (search!)
     ↓
2. AUGMENT: Stuff those documents into a prompt
     ↓
3. GENERATE: Ask an LLM to answer using the context
     ↓
Answer
```


---

# 1. Look at your data

In `data/fordham-website.zip` you'll find **~9,500 Markdown files** scraped from Fordham's website. Each file is one page — admissions info, program descriptions, faculty pages, financial aid, campus life, and more.

Your task: **look at the data**
- The first step in any AI engineering or data science project should always be to familiarize yourself with the data.
- I cannot stress this enough.. without this step, it's hard to build anything useful.

Tips:
- Unzip the archive and look at some of the files. 
- Open a few in a text editor. 
- Get a feel for what you're working with.
- The first line of every file is always the **URL** of the page it was scraped from. The rest is the page content converted to Markdown. Here's an example — `gabelli-school-of-business_veterans.md`:

```markdown
https://www.fordham.edu/gabelli-school-of-business/veterans

# Military Veterans & Active Duty Members of the Military

## Transform Your Knowledge & Skills Into a Business Career for the Future

As a veteran or an active duty member of the United States Armed Services,
you have gained or are currently acquiring the invaluable organizational,
leadership, analytics, and technical knowledge and skills that hiring
managers seek. These transferrable skills provide a major advantage in
emerging, business-related industries where innovation, a global mind-set,
and the ability to lead individuals and teams in the continuously evolving
work environment, are critical for success.

By completing a graduate or undergraduate business degree at the Gabelli
School of Business, you can prepare for a lifelong career in some of
today's fastest-growing fields. ...

### Study at a Top-Ranked, Military-Friendly University

The Gabelli School of Business is part of Fordham University, the only
New York City university to be among those ranked "Best for Vets" by
Military Times. ...

### Learn How the Yellow Ribbon Program Works

The Yellow Ribbon GI Education Enhancement Program, or the Yellow Ribbon
Program, is a part of the Post-9/11 Veterans Educational Assistance Act
of 2008. ...
```

The filenames mirror the URL structure — underscores replace path separators (e.g. `gabelli-school-of-business_veterans.md` came from `/gabelli-school-of-business/veterans`). Some files are short (a few lines), others are quite long.

- Once you've looked around, load the files into Python. Python's built-in `zipfile` module can read zip archives without extracting to disk. Load them into a list of dictionaries or a DataFrame with at least two fields: the filename (or a clean page name) and the content

In [2]:
import os
import pandas as pd
from pathlib import Path

# Path to your data folder (adjust this to match your setup)
DATA_DIR = 'data'  # or 'data/fordham-website' - whatever folder has the .md files

# Load all markdown files
documents = []

for md_file in Path(DATA_DIR).rglob('*.md'):  # rglob finds files in subfolders too
    with open(md_file, 'r', encoding='utf-8') as f:
        content = f.read()
        
    # Split into lines
    lines = content.split('\n')
    
    # First line is URL, rest is content
    url = lines[0].strip() if lines else ""
    page_content = '\n'.join(lines[1:]).strip()
    
    if page_content:  # Skip empty files
        documents.append({
            'filename': md_file.name,
            'url': url,
            'content': page_content,
            'length': len(page_content)
        })

print(f"Loaded {len(documents)} documents")

# Show some stats
df = pd.DataFrame(documents)
print(f"\nContent length stats:")
print(df['length'].describe())

Loaded 9530 documents

Content length stats:
count      9530.000000
mean       4184.807660
std        9023.852434
min         103.000000
25%        1268.250000
50%        2270.000000
75%        4370.750000
max      431113.000000
Name: length, dtype: float64


In [3]:
# Look at some example documents
import random

print("=" * 80)
print("EXAMPLE 1: Short document")
print("=" * 80)
short_doc = min(documents, key=lambda x: x['length'])
print(f"Filename: {short_doc['filename']}")
print(f"URL: {short_doc['url']}")
print(f"Length: {short_doc['length']} characters")
print(f"Content:\n{short_doc['content']}\n")

print("=" * 80)
print("EXAMPLE 2: Long document")
print("=" * 80)
long_doc = max(documents, key=lambda x: x['length'])
print(f"Filename: {long_doc['filename']}")
print(f"URL: {long_doc['url']}")
print(f"Length: {long_doc['length']} characters")
print(f"Content preview:\n{long_doc['content'][:500]}...\n")

print("=" * 80)
print("EXAMPLE 3: Random document")
print("=" * 80)
random_doc = random.choice(documents)
print(f"Filename: {random_doc['filename']}")
print(f"URL: {random_doc['url']}")
print(f"Length: {random_doc['length']} characters")
print(f"Content preview:\n{random_doc['content'][:500]}...")

EXAMPLE 1: Short document
Filename: graduate-school-of-education_academics_degree-and-certificate-programs_become-a-teacher_certificates-and-extensions.md
URL: https://www.fordham.edu/graduate-school-of-education/academics/degree-and-certificate-programs/become-a-teacher/certificates-and-extensions
Length: 103 characters
Content:
We offer many programs for certified teachers seeking extensions in their existing certification areas.

EXAMPLE 2: Long document
Filename: graduate-school-of-education_centers-and-institutes_the-center-for-educational-partnerships_nyc-regional-bilingual-education-resource-network_rbern-professional-opportunities--events.md
URL: https://www.fordham.edu/graduate-school-of-education/centers-and-institutes/the-center-for-educational-partnerships/nyc-regional-bilingual-education-resource-network/rbern-professional-opportunities--events
Length: 431113 characters
Content preview:
# RBERN Professional Opportunities & Events

**Note**: NYC RBERN at Fordham University 

---

# 2. Chunk the Documents

Some of the pages could be too long to embed as a single unit. Down the line, the pages may be too long to stuff into the LLM's prompt during the generation step. As such, most of the RAG systems will break down big documents into into smaller **chunks**.

> 📚 **TERM: Chunking**  
> Splitting documents into smaller, self-contained pieces for embedding and retrieval. The goal is chunks that are small enough to be specific, but large enough to be meaningful.

Your task: **write a function that splits each document into chunks.**

Things to think about:
- What's a reasonable chunk size? (Think about what fits in a prompt vs. what's too vague)
- Should you split on sentences? Paragraphs? A fixed character/word count?
- Should chunks overlap? What happens if an answer spans two chunks?
- How do you keep track of which document each chunk came from? You may need that information down the line.

In [6]:
# Simple chunking - no fancy libraries needed
all_chunks = []
chunk_id = 0

print("Creating chunks...")
for i, doc in enumerate(documents):
    if i % 1000 == 0:  # Progress update every 1000 docs
        print(f"Processing document {i}/{len(documents)}...")
    
    content = doc['content']
    
    # Simple: if document is short, keep it as one chunk
    if len(content) <= 1000:
        all_chunks.append({
            'chunk_id': chunk_id,
            'doc_filename': doc['filename'],
            'doc_url': doc['url'],
            'text': content
        })
        chunk_id += 1
    else:
        # Split long documents every 1000 characters at word boundaries
        start = 0
        while start < len(content):
            end = start + 1000
            if end < len(content):
                # Find last space before 1000 chars
                last_space = content[start:end].rfind(' ')
                if last_space > 0:
                    end = start + last_space
            
            chunk = content[start:end].strip()
            if chunk:
                all_chunks.append({
                    'chunk_id': chunk_id,
                    'doc_filename': doc['filename'],
                    'doc_url': doc['url'],
                    'text': chunk
                })
                chunk_id += 1
            
            start = end

print(f"\nDone! Created {len(all_chunks)} chunks")

Creating chunks...
Processing document 0/9530...
Processing document 1000/9530...
Processing document 2000/9530...
Processing document 3000/9530...
Processing document 4000/9530...
Processing document 5000/9530...
Processing document 6000/9530...
Processing document 7000/9530...
Processing document 8000/9530...
Processing document 9000/9530...

Done! Created 45020 chunks


In [7]:
# Check the chunks
print("Chunk statistics:")
chunk_lengths = [len(c['text']) for c in all_chunks]
print(f"Total chunks: {len(all_chunks)}")
print(f"Average chunk length: {sum(chunk_lengths) / len(chunk_lengths):.1f} characters")
print(f"Min: {min(chunk_lengths)}, Max: {max(chunk_lengths)}")

# Look at a few examples
print("\n" + "="*80)
print("Example chunks:")
print("="*80)
for i in range(3):
    chunk = all_chunks[i]
    print(f"\nChunk {i+1}:")
    print(f"From: {chunk['doc_url']}")
    print(f"Length: {len(chunk['text'])} chars")
    print(f"Text preview: {chunk['text'][:300]}...")
    print("-"*80)

Chunk statistics:
Total chunks: 45020
Average chunk length: 885.1 characters
Min: 2, Max: 1000

Example chunks:

Chunk 1:
From: https://www.fordham.edu/about/living-the-mission/campus-ministry/catholic-life/ministry-of-music
Length: 996 chars
Text preview: # Ministry of Music


Fordham offers each student a wealth of musical opportunities. Not only can students hear great music in the concert halls and churches of Manhattan but also experience this music firsthand through participation in a university choir. Whether concert oriented, or for the worshi...
--------------------------------------------------------------------------------

Chunk 2:
From: https://www.fordham.edu/about/living-the-mission/campus-ministry/catholic-life/ministry-of-music
Length: 992 chars
Text preview: Lessons and Carols (with the University Choir), the Holy Week Liturgies, and the Baccalaureate Mass. Members of the choir often serve as cantors and leaders of song. Membership is by audition. Undergraduate studen

---

# 3. Embed the Chunks

Now we need to turn each chunk into a vector so we can search over them. You've done this before in Lecture 4.

Your task: **embed all chunks using an embedding model.**

Tips:
- You could use a local model, or API model. What are the tradeoffs?
- This will take a while if you do it serially. You might want to use async/batch.
- Once you've created your embeddings, you may want to save them to disk so you don't have to redo this step every time
- You'll need to embed queries with the **same model** at search time

In [8]:
from sentence_transformers import SentenceTransformer
import numpy as np

print("Loading embedding model...")
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
print("Model loaded!")

# Extract just the text from chunks
chunk_texts = [chunk['text'] for chunk in all_chunks]

print(f"\nCreating embeddings for {len(chunk_texts)} chunks...")
print("This will take a few minutes...")

# Create embeddings in batches to avoid memory issues
embeddings = model.encode(
    chunk_texts, 
    show_progress_bar=True,
    batch_size=32
)

print(f"\nDone! Created embeddings with shape: {embeddings.shape}")

# Save embeddings so we don't have to recreate them
np.save('embeddings.npy', embeddings)
print("Saved embeddings to embeddings.npy")

/Users/suhaniwadhwa/ai-engineering-fordham/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading embedding model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1643.29it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded!

Creating embeddings for 45020 chunks...
This will take a few minutes...


Batches: 100%|██████████| 1407/1407 [03:48<00:00,  6.15it/s]



Done! Created embeddings with shape: (45020, 384)
Saved embeddings to embeddings.npy


---

# 4. Retrieve

Now build the **R** in RAG. Given a user's question, find the most relevant chunks.

Your task: **write a retrieval function that takes a question and returns the most relevant chunks.**

Tips:
- You can use lexical or semantic search or both!
- How many chunks should you retrieve? Too few and you might miss the answer; too many and you'll overwhelm the LLM (and pay more tokens)
- Try a few test questions and eyeball whether the retrieved chunks are relevant
- Try a few questions and see what comes back. For example:
  - "What programs does the Gabelli School of Business offer?"
  - "How do I apply for financial aid?"
  - "Where is Fordham's campus?"

In [10]:
from sklearn.metrics.pairwise import cosine_similarity

def retrieve_relevant_chunks(question, top_k=5):
    """
    Find the most relevant chunks for a question.
    
    Args:
        question: The user's question
        top_k: Number of chunks to retrieve
    
    Returns:
        List of relevant chunks with their similarity scores
    """
    # Embed the question using the same model
    question_embedding = model.encode([question])
    
    # Compute similarity with all chunks
    similarities = cosine_similarity(question_embedding, embeddings)[0]
    
    # Get top k indices
    top_indices = np.argsort(similarities)[-top_k:][::-1]
    
    # Return chunks with scores
    results = []
    for idx in top_indices:
        results.append({
            'chunk': all_chunks[idx],
            'similarity': float(similarities[idx])
        })
    
    return results

print("Retrieval function ready!")



Retrieval function ready!


In [11]:
# Test it with a question
test_question = "What programs does the Gabelli School of Business offer?"
print(f"\nTest question: {test_question}")
print("\nTop 3 results:")

results = retrieve_relevant_chunks(test_question, top_k=3)

for i, result in enumerate(results, 1):
    chunk = result['chunk']
    score = result['similarity']
    print(f"\n[{i}] Similarity: {score:.3f}")
    print(f"Source: {chunk['doc_url']}")
    print(f"Text preview: {chunk['text'][:200]}...")


Test question: What programs does the Gabelli School of Business offer?

Top 3 results:

[1] Similarity: 0.775
Source: https://www.fordham.edu/academics/colleges-and-schools/graduate-schools
Text preview: promote social justice, ethics in business, respect for the environment, and the defense of human rights around the globe.

### Gabelli School of Business

![Careers in Marketing - Business Developmen...

[2] Similarity: 0.766
Source: https://www.fordham.edu/gabelli-school-of-business/academic-programs-and-admissions
Text preview: # Gabelli Academic Programs and Admissions

Our wide-ranging business school programs won’t just shape your career – they’ll offer a transformative experience that will leave you ready to use business...

[3] Similarity: 0.753
Source: https://www.fordham.edu/gabelli-school-of-business/academic-programs-and-admissions/graduate-programs/academic-programs/phd-program
Text preview: # Ph.D. Program - Gabelli School of Business

![GSB PhD Program](/media/review/c

---

# 5. Generate

Now build the **G** in RAG. Take the retrieved chunks and pass them to an LLM along with the user's question.

Your task: **write a function that takes a question and the retrieved chunks, builds a prompt, and calls an LLM to generate an answer.**

Tips:
- How should you structure the prompt? The LLM needs to know: (1) what is the context of the application, (2) what is the question, (3) what it should include in its answer
- What should the LLM do if the context doesn't contain the answer?
- Start with a cheap model; try a better one when you've figured out the pipeline

In [12]:
from openai import OpenAI
from dotenv import load_dotenv
import os

# Load environment variables from .env file
load_dotenv()

# Initialize OpenAI client
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

def generate_answer(question, relevant_chunks):
    """
    Generate an answer using GPT based on retrieved chunks.
    
    Args:
        question: The user's question
        relevant_chunks: List of relevant chunks from retrieval
    
    Returns:
        Generated answer
    """
    # Build context from chunks
    context_parts = []
    for i, result in enumerate(relevant_chunks, 1):
        chunk = result['chunk']
        context_parts.append(f"[Source {i}: {chunk['doc_url']}]\n{chunk['text']}")
    
    context = "\n\n---\n\n".join(context_parts)
    
    # Create the prompt
    system_prompt = "You are a helpful assistant answering questions about Fordham University. Use only the provided context to answer questions. If the context doesn't contain enough information, say so honestly."
    
    user_prompt = f"""CONTEXT:
{context}

QUESTION: {question}

Please provide a helpful, accurate answer based on the context above. Mention which sources you used."""
    
    # Call OpenAI API
    response = client.chat.completions.create(
        model="gpt-4o-mini",  # Using the cheaper model
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.7,
        max_tokens=500
    )
    
    return response.choices[0].message.content

print("Generation function ready!")

# Test it
test_question = "What programs does the Gabelli School of Business offer?"
print(f"\nQuestion: {test_question}\n")

# Retrieve relevant chunks
relevant_chunks = retrieve_relevant_chunks(test_question, top_k=5)

# Generate answer
answer = generate_answer(test_question, relevant_chunks)

print("Answer:")
print(answer)

Generation function ready!

Question: What programs does the Gabelli School of Business offer?

Answer:
The Gabelli School of Business offers a variety of graduate and executive programs, including:

1. **Three types of M.B.A. programs**:
   - Full-time M.B.A.
   - Professional M.B.A.
   - Executive M.B.A.

2. **12 M.S. programs** (with two offered online).

3. **Two doctoral programs**:
   - Ph.D.
   - Doctor of Professional Studies.

Additionally, the school provides a comprehensive dual core curriculum that combines an integrated business core with a liberal arts core for all students, regardless of their eventual major or specialization. 

These details are sourced from [Source 1](https://www.fordham.edu/academics/colleges-and-schools/graduate-schools) and [Source 2](https://www.fordham.edu/gabelli-school-of-business/academic-programs-and-admissions).


---

# 6. Wire everything together

Combine the previous steps into a simple function that takes in a question and returns an answer.

Your task: **write a `rag(question)` function that retrieves relevant chunks and generates an answer.**

In [13]:
def rag(question, top_k=5, verbose=True):
    """
    Complete RAG pipeline: Retrieve relevant chunks and generate an answer.
    
    Args:
        question: The user's question
        top_k: Number of chunks to retrieve
        verbose: Whether to print intermediate steps
    
    Returns:
        The generated answer
    """
    if verbose:
        print("Retrieving relevant chunks...")
    
    # Step 1: Retrieve
    relevant_chunks = retrieve_relevant_chunks(question, top_k=top_k)
    
    if verbose:
        print(f"Found {len(relevant_chunks)} chunks")
        for i, result in enumerate(relevant_chunks, 1):
            print(f"  [{i}] {result['chunk']['doc_url'][:70]}... (similarity: {result['similarity']:.3f})")
        print("\nGenerating answer...\n")
    
    # Step 2: Generate
    answer = generate_answer(question, relevant_chunks)
    
    return answer

print("Complete RAG pipeline ready!\n")

# Test with multiple questions
test_questions = [
    "How do I apply for undergraduate admission?",
    "What financial aid options are available?",
    "What is the Yellow Ribbon Program?"
]

for question in test_questions:
    print("="*80)
    print(f"Question: {question}")
    print("="*80)
    
    answer = rag(question, top_k=5, verbose=False)
    
    print(f"\nAnswer:\n{answer}\n")
    

Complete RAG pipeline ready!

Question: How do I apply for undergraduate admission?

Answer:
The provided context does not contain specific information about applying for undergraduate admission to Fordham University. The sources primarily discuss graduate programs and their admission requirements, including details about transcripts and recommendation letters for master's programs. Therefore, I cannot provide an answer regarding undergraduate admission based on the available context.

Question: What financial aid options are available?

Answer:
At Fordham University, there are several financial aid options available for graduate students:

1. **Federal Financial Aid**: Students can apply for federal aid by completing the FAFSA. This typically includes:
   - Federal Direct Unsubsidized Loans
   - Federal Direct Graduate PLUS Loans
   - Private Education Loans

   For more information, students can contact the Office of Student Financial Services at 718-817-3800 or via email at [email p

---

# 7. Evaluate, experiment and improve

Your RAG system works — but there's always room to make it better. 

Your task: **evaluate, experiment, and improve your system**

Tips:
- How do you know that your system is working or that your changes are improving it?
- Try different questions — where does it do well? Where does it struggle?
- Adjust the number of retrieved chunks — what happens with more or fewer?
- Try different chunking strategies — bigger chunks? Smaller? Overlap?
- Try a different embedding model — does it change retrieval quality?
- Improve the prompt — can you get better, more concise answers?
- Add source attribution — can the system tell the user which pages the answer came from?

In [14]:
# Interactive testing - ask your own questions!

print("Fordham RAG System - Interactive Mode")
print("="*80)
print("Ask questions about Fordham University")
print("Type 'quit' to stop\n")

while True:
    question = input("Your question: ")
    
    if question.lower() in ['quit', 'exit', 'q']:
        print("Goodbye!")
        break
    
    if not question.strip():
        continue
    
    print("\n" + "="*80)
    answer = rag(question, top_k=5, verbose=True)
    print(f"\nAnswer:\n{answer}")
    print("="*80 + "\n")

Fordham RAG System - Interactive Mode
Ask questions about Fordham University
Type 'quit' to stop


Retrieving relevant chunks...
Found 5 chunks
  [1] https://www.fordham.edu/information-technology/standard-software/micro... (similarity: 0.801)
  [2] https://www.fordham.edu/academics/faculty/faculty-senate/faculty-senat... (similarity: 0.739)
  [3] https://www.fordham.edu/gabelli-school-of-business/academic-programs-a... (similarity: 0.720)
  [4] https://www.fordham.edu/about/campuses/rose-hill-campus... (similarity: 0.717)
  [5] https://www.fordham.edu/about/leadership-and-administration/administra... (similarity: 0.713)

Generating answer...


Answer:
Fordham University has its Rose Hill campus located in the Bronx, New York. This information is sourced from Source 4, which provides details about the campus.


Retrieving relevant chunks...
Found 5 chunks
  [1] https://www.fordham.edu/academics/research/office-of-sponsored-program... (similarity: 0.563)
  [2] https://www.fordham.edu/ab

---

# 8. (Optional) Make it an app

So far your RAG system lives inside a notebook. That's great for development — but nobody is going to use your Jupyter notebook to ask questions about Fordham. Let's turn it into a real web app.

> 📚 **TERM: Streamlit**  
> A Python library that turns plain Python scripts into interactive web apps. You write Python — no HTML, CSS, or JavaScript — and Streamlit renders it as a web page with inputs, buttons, and formatted output. It's the fastest way to go from "I have a function" to "I have a web app."

Your task: **create a Streamlit app that lets a user type a question about Fordham and get an answer from your RAG system.**

To get started:
- Install it: `uv pip install streamlit` 
- A Streamlit app is just a `.py` file (not a notebook). Create something like `fordham_rag_app.py`
- Run it: `streamlit run scripts/fordham_rag_app.py` — this opens a browser tab with your app

Tips:
- Check out the [Streamlit docs](https://docs.streamlit.io/) — the "Get started" tutorial is very short
- Your best bet is to vibecode your way to this. You'll be surprised how fast you can get it up and running

In [15]:
import json

# Save chunks to file for the Streamlit app
with open('chunks.json', 'w') as f:
    json.dump(all_chunks, f)

print("Saved chunks to chunks.json")

Saved chunks to chunks.json


---

# Summary

## What You Built

| Step | What You Did | What It Does |
|------|-------------|-------------|
| **Load** | Read 9,500+ Fordham web pages | Get raw content |
| **Chunk** | Split pages into smaller pieces | Make content searchable and promptable |
| **Embed** | Turn chunks into vectors | Enable semantic search |
| **Retrieve** | Find relevant chunks for a question | The **R** in RAG |
| **Generate** | Ask an LLM to answer using the chunks | The **G** in RAG |
| **RAG** | Wire it all together | Question in, answer out |

## The Big Picture

RAG is one of the most common patterns in AI engineering today. What you built here is the same core architecture behind tools like ChatGPT with search, Perplexity, enterprise Q&A bots, and more. The details get more sophisticated (vector databases, reranking, query rewriting, evaluation) but the pattern is the same:

**Find relevant stuff → give it to an LLM → get an answer.**

You can just build things.